### Imports

In [1]:
import os
import sys
import json
import time
import argparse
import warnings
import xml.etree.ElementTree as ET
import multiprocessing as mp
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
from PIL import Image, ImageDraw

import daisy
import dask
from dask.array import coarsen, mean
from dask.diagnostics import ProgressBar
import zarr
from funlib.persistence import Array, prepare_ds, open_ds
from funlib.geometry import Roi, Coordinate
import tifffile
from tqdm import tqdm
from skimage.measure import label, regionprops
from scipy import ndimage

import rtree
from shapely.geometry import Polygon, box

# for xml > zarr
from shapely.geometry import Polygon, box
from shapely.ops import unary_union
from shapely.strtree import STRtree
from skimage.draw import polygon as draw_polygon

c:\Users\Waluigi\anaconda3\envs\transunet_dataprep\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))
grandparent_dir = os.path.abspath(os.path.join(cwd, os.pardir, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(grandparent_dir, 'openslide-bin-4.0.0.11-windows-x64\\bin')
print(OPENSLIDE_PATH)
if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

c:\Users\Waluigi\Desktop\github_repos\DCIS\openslide-bin-4.0.0.11-windows-x64\bin


In [4]:
def opensvs(svs_path, pyramid_level): # from rachel
    """
    open .svs as a dask array and retreive metadata (dimensions + resolution)
    
    INPUTS: 
    - svs_path (str): path to .svs file
    - pyramid_level (int): which pyramid level to open (0 = 40x, 1 = 20x, 2 = 10x, 3 = 5x)
    
    OUTPUTS:
    - dask_array: dask array of the image data for the specified pyramid level
    - x_res: resolution in nm/px in the x dimension
    - y_res: resolution in nm/px in the y dimension
    - units: tuple of units for x and y resolution (should be ("nm", "nm"))
    """
    slide = openslide.OpenSlide(svs_path)
    # grab resolution at micrometeres / px and convert to nm / px
    x_res = float(slide.properties["openslide.mpp-x"]) * 1000
    y_res = float(slide.properties["openslide.mpp-y"]) * 1000
    units = ("nm", "nm")
    # read pyramid level directly — no zarr involved
    with tifffile.TiffFile(svs_path) as tif:
        level = tif.series[0].levels[pyramid_level]
        dask_array = dask.array.from_array(level.asarray(), chunks='auto')

    return dask_array, x_res, y_res, units

def svs_to_zarr(svs_path, zarr_path, offset, axis_names): # from rachel
    """
    convert H&E from .svs to .zarr file

    INPUTS:
    - svs_path (str): path to .svs file 
    - zarr_path (str): path to save .zarr file
    - offset (tuple): offset for the image data (e.g. (0, 0) if no offset)
    - axis_names (tuple): names of the axes (e.g. ("x", "y", "c"))

    OUTPUTS: 
    - saves .zarr file with the image data for each pyramid level (s0 = 40x, s1 = 20x, s2 = 10x, s3 = 5x) and metadata (voxel size, axis names, units)
    """
    # open highest pyramid level (40x)
    dask_array0, x_res, y_res, units = opensvs(svs_path, 0)
    s0_shape = dask_array0.shape
    # units are natively ("nm", "nm") so no need to convert to get voxel size
    
    # convert to integer and calculate for each pyramid level
    voxel_size0 = Coordinate(int(x_res), int(y_res))
    
    # format data as funlib dataset
    raw = prepare_ds(
        zarr_path / "raw" / "s0",
        dask_array0.shape,
        offset,
        voxel_size0,
        axis_names,
        units,
        mode="w",
        dtype=np.uint8,
    )
    
    # storage info
    store_rgb = zarr.open(zarr_path / "raw" / "s0") # s0 = full resolution image at 40x magnification
    dask_array = dask_array0.rechunk(raw.data.chunksize)

    with ProgressBar():
        dask.array.store(dask_array, store_rgb)

    for i in range(1, 4): # iterate over each pyramid level: s1 (20x), s2 (10x), s3 (5x)
        # open the image file with openslide for info and tifffile as zarr
        try:
            dask_array, x_res, y_res, _ = opensvs(svs_path, i)
            # units are natively ("nm", "nm") so no need to convert to get voxel size
            
            # convert to integer and calculate for each pyramid level
            voxel_size0 = Coordinate(int(x_res), int(y_res))
            expected_shape = tuple((s0_shape[0] // 2**i, s0_shape[1] // 2**i, 3)) # downsampled shape
            print(f"expected shape: {expected_shape}")
            print(f"actual shape: {dask_array.shape}")

            # check shape is expected shape
            if dask_array.shape == expected_shape:
                print("correct shape")
                
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    dask_array.shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                dask_array = dask_array.rechunk(raw.data.chunksize)

                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)

            else:
                voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
                # format data as funlib dataset
                raw = prepare_ds(
                    zarr_path / "raw" / f"s{i}",
                    expected_shape,
                    offset,
                    voxel_size,
                    axis_names,
                    units,
                    mode="w",
                    dtype=np.uint8,
                )
                # storage info
                store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
                prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
                print(f"chunk shape: {prev_layer.chunk_shape}")

                # mean downsampling
                factors = {0: 2, 1: 2}
                try:
                    dask_array = coarsen(mean, prev_layer.data, factors)
                except ValueError as e:
                    new_shape = tuple(
                        (
                            (prev_layer.data.shape[i] // factors[i]) * factors[i]
                            if i in factors
                            else prev_layer.data.shape[i]
                        )
                        for i in range(prev_layer.data.ndim)
                    )
                    dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                    dask_array = coarsen(mean, dask_array_cropped, factors)
                # save to zarr
                with ProgressBar():
                    dask.array.store(dask_array, store_rgb)
        
        except TypeError as e: # if it finds an empty pyramid level, it fills it in
            print(f"for layer {i}: {e}")
            print("Generating layer")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            voxel_size = tuple((voxel_size0[0] * 2**i, voxel_size0[0] * 2**i))
            # format data as funlib dataset
            raw = prepare_ds(
                zarr_path / "raw" / f"s{i}",
                expected_shape,
                offset,
                voxel_size,
                axis_names,
                units,
                mode="w",
                dtype=np.uint8,
            )
            # storage info
            store_rgb = zarr.open(zarr_path / "raw" / f"s{i}")
            prev_layer = open_ds(zarr_path / "raw" / f"s{i-1}")
            print(f"chunk shape: {prev_layer.chunk_shape}")
            # mean downsampling
            factors = {0: 2, 1: 2}
            try:
                dask_array = coarsen(mean, prev_layer.data, factors)
            except ValueError as e:
                new_shape = tuple(
                    (
                        (prev_layer.data.shape[i] // factors[i]) * factors[i]
                        if i in factors
                        else prev_layer.data.shape[i]
                    )
                    for i in range(prev_layer.data.ndim)
                )
                dask_array_cropped = prev_layer.data[:new_shape[0], :new_shape[1], :new_shape[2]]
                dask_array = coarsen(mean, dask_array_cropped, factors)
            # save to zarr
            with ProgressBar():
                dask.array.store(dask_array, store_rgb)
    return print("svs conversion complete")

In [ ]:
# --- convert .svs to .zarr
svs_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283.svs")
zarr_path = Path(r"E:\[PROJ]_DCIS\BRACS\BRACS_1283.zarr")
offset = Coordinate(0, 0)
axis_names = ['y', 'x', 'c^']

svs_to_zarr(svs_path, 
            zarr_path, 
            offset, 
            axis_names 
            )

[########################################] | 100% Completed | 192.76 s
expected shape: (39928, 88644, 3)
actual shape: (19964, 44322, 3)
chunk shape: (1248, 5541, 1)
[########################################] | 100% Completed | 247.07 s
expected shape: (19964, 44322, 3)
actual shape: (4991, 11080, 3)
chunk shape: (1248, 2771, 1)
[########################################] | 100% Completed | 116.88 s
expected shape: (9982, 22161, 3)
actual shape: (1247, 2770, 3)
chunk shape: (1248, 2771, 1)
[########################################] | 100% Completed | 18.79 s
svs conversion complete
